# Week 7 — File Handling & Error Handling
### Python for Blockchain Analytics | Phase 2

---

**What you'll learn this week:**

- Reading and writing **CSV files** — the most common format for blockchain data exports
- Reading and writing **JSON files** — the native format of every blockchain API
- **Error handling** with `try / except / finally` — writing code that doesn't crash
- **Custom exceptions** — building your own error types for blockchain-specific problems
- **Logging** — replacing `print()` with proper logging for production code
- **Real free data sources** — where to get on-chain data without paying for API keys

**Why this week matters:**

Every piece of blockchain data you'll ever analyse arrives as a file or an API response.
Before you can do anything useful with it, you need to read it reliably, handle bad data
gracefully, and know what to do when things go wrong — because in blockchain analytics,
things go wrong constantly: RPC nodes time out, APIs return unexpected shapes,
CSV exports have missing columns, block reorgs corrupt your data.

This week teaches you the defensive programming habits that separate a script
that works once from a pipeline that runs reliably every day.

**SQL analyst parallel:**
- Reading a CSV = `COPY FROM` or bulk import
- Error handling = `TRY...CATCH` in a stored procedure
- Logging = SQL Server trace or PostgreSQL `log_min_messages`

**Free data sources used this week (no API key required):**
- **CoinGecko** — `/coins/markets` endpoint, free, no key
- **DefiLlama** — `https://api.llama.fi/protocols`, completely free
- **Etherscan** — free tier with a free API key (takes 30 seconds to register)
- **Ankr public RPC** — `https://rpc.ankr.com/eth`, free, no signup

**Time:** ~3.5 hours

---

## 1. CSV Files — Reading and Writing Blockchain Data

CSV (Comma-Separated Values) is the simplest and most universal data format.
When you export data from Etherscan, Dune, or any analytics tool, you almost always
get a CSV. When you want to share data or load it into Pandas, CSV is the starting point.

Python's built-in `csv` module handles reading and writing without any installation.

### 1.1 Writing CSV files — saving your analysis

In [ ]:
import csv
import os

# Sample data — what you'd have after analysing on-chain transactions
transactions = [
    {"tx_hash": "0xaaa111", "from": "0xAlice", "to": "0xBob",   "value_eth": 1.5,  "gas_gwei": 20, "status": "success", "block": 19_847_293},
    {"tx_hash": "0xbbb222", "from": "0xBob",   "to": "0xCarol", "value_eth": 0.3,  "gas_gwei": 22, "status": "success", "block": 19_847_294},
    {"tx_hash": "0xccc333", "from": "0xAlice", "to": "0xDave",  "value_eth": 5.0,  "gas_gwei": 18, "status": "failed",  "block": 19_847_295},
    {"tx_hash": "0xddd444", "from": "0xCarol", "to": "0xAlice", "value_eth": 0.05, "gas_gwei": 25, "status": "success", "block": 19_847_296},
    {"tx_hash": "0xeee555", "from": "0xDave",  "to": "0xBob",   "value_eth": 12.0, "gas_gwei": 30, "status": "success", "block": 19_847_297},
]

# Writing a CSV file
# newline='' is important on Windows — prevents extra blank lines
output_path = "transactions.csv"

with open(output_path, "w", newline="") as f:
    # fieldnames defines the column order
    fieldnames = ["tx_hash", "from", "to", "value_eth", "gas_gwei", "status", "block"]

    writer = csv.DictWriter(f, fieldnames=fieldnames)

    writer.writeheader()          # writes the header row
    writer.writerows(transactions) # writes all rows at once

print(f"Written {len(transactions)} rows to {output_path}")

# Verify by reading it back as raw text
with open(output_path) as f:
    print("\nFile contents:")
    print(f.read())

In [ ]:
# Reading a CSV file — the most common operation in data analytics

with open("transactions.csv", newline="") as f:
    reader = csv.DictReader(f)    # each row becomes a dict with column names as keys

    rows = list(reader)           # read all rows into a list

print(f"Read {len(rows)} rows")
print(f"Columns: {list(rows[0].keys())}")
print(f"\nFirst row: {rows[0]}")
print(f"Type of value_eth: {type(rows[0]['value_eth'])}")

# ⚠️  CRITICAL: CSV always reads everything as strings — you MUST cast
# This is a very common bug for beginners
print("\n⚠️  Everything is a string from CSV — cast before arithmetic:")
for row in rows:
    value_eth = float(row["value_eth"])    # cast string → float
    block     = int(row["block"])          # cast string → int
    is_ok     = row["status"] == "success" # string comparison is fine

    print(f"  Block {block:,} | {value_eth:.4f} ETH | Success: {is_ok}")

In [ ]:
# A production-ready CSV reader function
# Handles: type casting, missing values, malformed rows

def read_transaction_csv(filepath: str) -> list:
    """
    Read a transaction CSV file and return a list of parsed dicts.

    Handles type casting and missing/malformed values gracefully.

    Args:
        filepath: Path to the CSV file

    Returns:
        list: Parsed transaction dicts with correct Python types
    """
    transactions = []
    skipped      = 0

    with open(filepath, newline="") as f:
        reader = csv.DictReader(f)

        for i, row in enumerate(reader, start=2):  # start=2 because row 1 is the header
            try:
                transactions.append({
                    "tx_hash":   row["tx_hash"].strip(),
                    "from":      row["from"].strip(),
                    "to":        row["to"].strip(),
                    "value_eth": float(row["value_eth"]),
                    "gas_gwei":  int(row["gas_gwei"]),
                    "status":    row["status"].strip(),
                    "block":     int(row["block"]),
                })
            except (ValueError, KeyError) as e:
                print(f"  ⚠️  Skipping row {i}: {e}")
                skipped += 1

    print(f"  Loaded {len(transactions)} rows, skipped {skipped} malformed rows")
    return transactions


txns = read_transaction_csv("transactions.csv")

# Now analyse the clean data
total_volume  = sum(t["value_eth"] for t in txns if t["status"] == "success")
success_count = sum(1 for t in txns if t["status"] == "success")
failed_count  = sum(1 for t in txns if t["status"] == "failed")

print(f"\n  Total volume:   {total_volume:.4f} ETH")
print(f"  Success:        {success_count}")
print(f"  Failed:         {failed_count}")

### 1.2 Writing analysis results back to CSV

A common pattern: read raw data → analyse → write results to a new CSV.

In [ ]:
# Aggregate by wallet and write summary to CSV
from collections import defaultdict

wallet_stats = defaultdict(lambda: {"sent": 0.0, "received": 0.0, "tx_count": 0})

for tx in txns:
    if tx["status"] != "success":
        continue
    wallet_stats[tx["from"]]["sent"]     += tx["value_eth"]
    wallet_stats[tx["from"]]["tx_count"] += 1
    wallet_stats[tx["to"]]["received"]   += tx["value_eth"]

# Convert to list of dicts and add net position
summary = []
for wallet, stats in wallet_stats.items():
    summary.append({
        "wallet":    wallet,
        "sent_eth":  round(stats["sent"], 6),
        "recv_eth":  round(stats["received"], 6),
        "net_eth":   round(stats["received"] - stats["sent"], 6),
        "tx_count":  stats["tx_count"],
    })

summary.sort(key=lambda r: -r["recv_eth"])

# Write summary CSV
summary_path = "wallet_summary.csv"
with open(summary_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["wallet","sent_eth","recv_eth","net_eth","tx_count"])
    writer.writeheader()
    writer.writerows(summary)

print(f"Wallet summary written to {summary_path}")
print()

# Preview
with open(summary_path) as f:
    print(f.read())

## 2. JSON Files — The Native Language of Blockchain APIs

Every blockchain API — Etherscan, CoinGecko, DefiLlama, The Graph — returns JSON.
Understanding how to read, write, and navigate JSON is the single most important
file skill for blockchain analytics.

JSON maps directly to Python data structures:

| JSON | Python |
|------|--------|
| `{}` object | `dict` |
| `[]` array | `list` |
| `"string"` | `str` |
| `123` / `1.5` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Python's built-in `json` module handles everything.

In [ ]:
import json

# Writing JSON — saving API responses or your own structured data
protocol_data = {
    "name":        "Uniswap V3",
    "chain":       "ethereum",
    "tvl_usd":     5_200_000_000,
    "volume_24h":  1_800_000_000,
    "top_pools": [
        {"pair": "USDC/ETH",  "fee": 500,  "tvl": 800_000_000},
        {"pair": "USDC/ETH",  "fee": 3000, "tvl": 450_000_000},
        {"pair": "ETH/USDT",  "fee": 3000, "tvl": 320_000_000},
    ],
    "is_verified": True,
    "launch_year": 2021,
}

# json.dump — write to a file
with open("protocol.json", "w") as f:
    json.dump(protocol_data, f, indent=2)   # indent=2 makes it human-readable

# json.dumps — write to a string (useful for APIs and logging)
json_string = json.dumps(protocol_data, indent=2)
print("First 200 chars of JSON string:")
print(json_string[:200])
print("...")

In [ ]:
# Reading JSON — loading API responses from disk

# json.load — read from a file
with open("protocol.json") as f:
    loaded = json.load(f)

print(f"Type: {type(loaded)}")
print(f"Protocol: {loaded['name']}")
print(f"TVL: ${loaded['tvl_usd']/1e9:.2f}B")
print(f"Top pools: {len(loaded['top_pools'])}")
print(f"Largest pool: {loaded['top_pools'][0]['pair']}")

# json.loads — parse a JSON string (what you get from requests.response.json())
raw_json_string = \'\'\'
{
    "status": "1",
    "message": "OK",
    "result": [
        {"blockNumber": "19847293", "hash": "0xabc...", "value": "1500000000000000000"},
        {"blockNumber": "19847294", "hash": "0xdef...", "value": "500000000000000000"}
    ]
}
\'\'\'

parsed = json.loads(raw_json_string)
print(f"\nAPI status: {parsed['status']}")
print(f"Transactions: {len(parsed['result'])}")
for tx in parsed["result"]:
    eth = int(tx["value"]) / 1e18
    print(f"  Block {tx['blockNumber']} | {eth:.4f} ETH")

In [ ]:
# Caching API responses to disk — avoid hitting rate limits during development
# This is a critical pattern: fetch once, save to disk, load from disk on reruns

import os
import time

def load_or_fetch(cache_path: str, fetch_fn, max_age_seconds: int = 3600):
    """
    Load data from cache if fresh enough, otherwise fetch and cache it.

    This pattern is essential for blockchain analytics:
    - APIs have rate limits (5 calls/sec on Etherscan free tier)
    - You don't want to re-fetch data you already have
    - During development you iterate on analysis, not data collection

    Args:
        cache_path:       Path to the JSON cache file
        fetch_fn:         Function to call if cache is stale/missing
        max_age_seconds:  How old the cache can be before refreshing (default: 1 hour)

    Returns:
        The cached or freshly fetched data
    """
    # Check if cache exists and is fresh
    if os.path.exists(cache_path):
        age = time.time() - os.path.getmtime(cache_path)
        if age < max_age_seconds:
            print(f"  📂 Loading from cache: {cache_path} ({age:.0f}s old)")
            with open(cache_path) as f:
                return json.load(f)
        else:
            print(f"  ⏰ Cache stale ({age:.0f}s old > {max_age_seconds}s) — refreshing")
    else:
        print(f"  🔍 No cache found — fetching fresh data")

    # Cache miss — fetch and save
    data = fetch_fn()

    with open(cache_path, "w") as f:
        json.dump(data, f, indent=2)

    print(f"  💾 Saved to cache: {cache_path}")
    return data


# Example usage (simulated fetch — real version in Week 10 with requests)
def simulated_fetch():
    """Simulate fetching from DefiLlama API."""
    return {
        "protocols": [
            {"name": "Lido",       "tvl": 32_000_000_000, "chain": "Ethereum"},
            {"name": "Aave V3",    "tvl": 12_800_000_000, "chain": "Ethereum"},
            {"name": "Uniswap V3", "tvl": 5_200_000_000,  "chain": "Ethereum"},
        ],
        "fetched_at": int(time.time())
    }

# First call — fetches and caches
data = load_or_fetch("defillama_cache.json", simulated_fetch, max_age_seconds=60)
print(f"  Protocols loaded: {len(data['protocols'])}")

# Second call — loads from cache
data2 = load_or_fetch("defillama_cache.json", simulated_fetch, max_age_seconds=60)
print(f"  Same data? {data == data2}")

## 3. Error Handling — Writing Code That Doesn't Crash

In blockchain analytics, things fail constantly:
- RPC nodes time out or return errors
- An API changes its response shape
- A CSV has a missing column or a malformed row
- A block reorg causes your data to be inconsistent
- You try to divide by zero on an empty pool

Without error handling, your entire pipeline crashes on the first problem.
With it, you can handle failures gracefully, log what went wrong, and keep running.

### 3.1 The try/except/else/finally structure

```
try:
    # code that might fail
except SomeError as e:
    # what to do when that specific error happens
except (AnotherError, YetAnother) as e:
    # catch multiple error types in one block
else:
    # runs only if NO exception was raised
finally:
    # ALWAYS runs, whether exception happened or not
    # use for cleanup: close files, release connections
```

**The critical rule:** catch the most specific exception you can.
Never use a bare `except:` — it hides bugs.


In [ ]:
# Understanding Python's exception hierarchy
# The errors you'll encounter most in blockchain analytics:

# ValueError     — wrong type of value (e.g. int("abc"))
# KeyError       — missing dict key (e.g. data["missing_key"])
# TypeError      — wrong type for an operation (e.g. "1" + 1)
# ZeroDivisionError — dividing by zero (e.g. tvl/users when users=0)
# FileNotFoundError — file doesn't exist
# json.JSONDecodeError — malformed JSON string
# IndexError     — list index out of range

# Let's see them in blockchain context

def parse_etherscan_tx(raw_tx: dict) -> dict:
    """
    Parse a raw Etherscan transaction dict into clean Python types.

    Demonstrates: KeyError when fields are missing, ValueError on bad casts.
    """
    try:
        return {
            "hash":      raw_tx["hash"],             # KeyError if missing
            "value_eth": int(raw_tx["value"]) / 1e18, # ValueError if not numeric
            "gas_used":  int(raw_tx["gasUsed"]),
            "block":     int(raw_tx["blockNumber"]),
            "success":   raw_tx["txreceipt_status"] == "1",
        }
    except KeyError as e:
        print(f"  ❌ Missing field in transaction: {e}")
        return None
    except ValueError as e:
        print(f"  ❌ Bad value in transaction: {e}")
        return None

# Test with good data
good_tx = {
    "hash": "0xabc123",
    "value": "1500000000000000000",
    "gasUsed": "21000",
    "blockNumber": "19847293",
    "txreceipt_status": "1",
}
result = parse_etherscan_tx(good_tx)
print(f"Good tx parsed: {result}")

# Test with missing field
bad_tx_1 = {"hash": "0xdef456", "value": "500000000000000000"}
result2 = parse_etherscan_tx(bad_tx_1)
print(f"Bad tx (missing fields): {result2}")

# Test with bad value
bad_tx_2 = {**good_tx, "value": "not_a_number"}
result3 = parse_etherscan_tx(bad_tx_2)
print(f"Bad tx (bad value): {result3}")

In [ ]:
# else and finally — the full structure

def read_json_file(filepath: str) -> dict:
    """
    Read a JSON file with proper error handling.

    Uses else (success path) and finally (cleanup) properly.
    """
    f = None
    try:
        f = open(filepath)
        data = json.load(f)

    except FileNotFoundError:
        # The file simply doesn't exist — clear, actionable message
        print(f"  ❌ File not found: {filepath}")
        return {}

    except json.JSONDecodeError as e:
        # File exists but JSON is malformed — tell them where
        print(f"  ❌ Invalid JSON in {filepath}: line {e.lineno}, col {e.colno}")
        return {}

    except PermissionError:
        # Rare but real — file exists but we can't read it
        print(f"  ❌ Permission denied: {filepath}")
        return {}

    else:
        # Only runs if NO exception occurred
        print(f"  ✅ Loaded {filepath} ({len(str(data))} bytes)")
        return data

    finally:
        # ALWAYS runs — even if we returned early above
        # Use for cleanup: close file handles, database connections, etc.
        if f:
            f.close()
            # (In practice: use 'with open()' which handles this automatically)


# Test each case
print("Reading valid JSON:")
data = read_json_file("protocol.json")

print("\nReading missing file:")
data = read_json_file("doesnt_exist.json")

print("\nReading invalid JSON:")
with open("bad.json", "w") as f:
    f.write("{ this is not valid json }")
data = read_json_file("bad.json")

In [ ]:
# Chaining multiple fallbacks — the retry and fallback pattern
# Critical for blockchain analytics: primary RPC might fail, fall back to secondary

def fetch_eth_balance(address: str, rpc_urls: list) -> float:
    """
    Try to fetch an ETH balance from multiple RPC endpoints.
    Falls back to the next one if a request fails.
    In practice you'd use requests or web3.py — here we simulate the logic.
    """
    last_error = None

    for i, rpc_url in enumerate(rpc_urls, 1):
        try:
            # Simulating an RPC call — in real code: web3.eth.get_balance(address)
            # Here we simulate: first RPC fails, second succeeds
            if i == 1:
                raise ConnectionError(f"RPC {rpc_url} timed out")

            # Simulated successful response
            balance_wei = 2_500_000_000_000_000_000
            balance_eth = balance_wei / 1e18
            print(f"  ✅ Got balance from RPC #{i}: {rpc_url}")
            return balance_eth

        except ConnectionError as e:
            print(f"  ⚠️  RPC #{i} failed: {e} — trying next...")
            last_error = e
            continue   # try the next RPC

    # All RPCs failed
    raise ConnectionError(f"All {len(rpc_urls)} RPCs failed. Last error: {last_error}")


FREE_RPC_ENDPOINTS = [
    "https://rpc.ankr.com/eth",          # Ankr — free, no signup
    "https://eth.llamarpc.com",           # LlamaNodes — free
    "https://ethereum.publicnode.com",    # PublicNode — free
]

try:
    balance = fetch_eth_balance("0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
                                FREE_RPC_ENDPOINTS)
    print(f"  ETH balance: {balance:.4f} ETH")
except ConnectionError as e:
    print(f"  ❌ Could not fetch balance: {e}")

## 4. Custom Exceptions — Building Your Own Error Types

Python's built-in exceptions (`ValueError`, `KeyError`) are generic.
When you're building a blockchain analytics tool, you want errors that
tell you *exactly* what went wrong in the blockchain context.

Custom exceptions make your code:
- **Self-documenting** — `InvalidAddressError` is clearer than `ValueError`
- **Catchable precisely** — callers can catch `BlockReorgError` without catching everything
- **Informative** — you can include address, block number, chain ID in the error

**Pattern:** create a base exception for your module, then subclass it.

In [ ]:
# Define a hierarchy of blockchain-specific exceptions

class BlockchainError(Exception):
    """Base exception for all blockchain analytics errors.
    All our custom errors inherit from this so callers can catch them all
    with a single 'except BlockchainError' if they want to.
    """
    pass


class InvalidAddressError(BlockchainError):
    """Raised when an Ethereum address is malformed."""

    def __init__(self, address: str, reason: str = ""):
        self.address = address
        self.reason  = reason
        msg = f"Invalid Ethereum address: {address!r}"
        if reason:
            msg += f" — {reason}"
        super().__init__(msg)


class InsufficientBalanceError(BlockchainError):
    """Raised when a wallet has insufficient funds for an operation."""

    def __init__(self, address: str, required: float, available: float, token: str = "ETH"):
        self.address   = address
        self.required  = required
        self.available = available
        self.token     = token
        super().__init__(
            f"Insufficient {token} for {address[:10]}...: "
            f"need {required:.6f}, have {available:.6f}"
        )


class APIError(BlockchainError):
    """Raised when an external API call fails."""

    def __init__(self, api_name: str, status_code: int = None, message: str = ""):
        self.api_name    = api_name
        self.status_code = status_code
        self.message     = message
        parts = [f"{api_name} API error"]
        if status_code:
            parts.append(f"(HTTP {status_code})")
        if message:
            parts.append(f": {message}")
        super().__init__(" ".join(parts))


class BlockReorgError(BlockchainError):
    """Raised when a block reorganisation is detected in the data."""

    def __init__(self, block_number: int, expected_hash: str, actual_hash: str):
        self.block_number  = block_number
        self.expected_hash = expected_hash
        self.actual_hash   = actual_hash
        super().__init__(
            f"Block reorg detected at #{block_number}: "
            f"expected {expected_hash[:10]}..., got {actual_hash[:10]}..."
        )


class RateLimitError(APIError):
    """Raised when an API rate limit is hit."""

    def __init__(self, api_name: str, retry_after: int = None):
        self.retry_after = retry_after
        msg = f"Rate limit exceeded"
        if retry_after:
            msg += f" — retry after {retry_after}s"
        super().__init__(api_name, status_code=429, message=msg)


# ── Using custom exceptions ────────────────────────────────────
def validate_address(address: str) -> str:
    """Validate an Ethereum address. Returns the address or raises InvalidAddressError."""
    if not isinstance(address, str):
        raise InvalidAddressError(str(address), "must be a string")
    if not address.startswith("0x"):
        raise InvalidAddressError(address, "must start with '0x'")
    if len(address) != 42:
        raise InvalidAddressError(address, f"must be 42 chars, got {len(address)}")
    hex_chars = set("0123456789abcdefABCDEF")
    if not all(c in hex_chars for c in address[2:]):
        raise InvalidAddressError(address, "contains non-hex characters")
    return address


# Test the custom exceptions
test_addresses = [
    "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",   # valid
    "d8dA6BF26964aF9D7eEd9e03E53415D37aA96045",     # missing 0x
    "0xd8dA6BF26964aF9D7eEd9e03E53",                # too short
    "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA9604Z",   # bad char
]

print("Address validation:")
for addr in test_addresses:
    try:
        validated = validate_address(addr)
        print(f"  ✅ {addr[:20]}... — valid")
    except InvalidAddressError as e:
        print(f"  ❌ {e}")

# Catching the base class catches all subclasses
print("\nCatching all blockchain errors:")
try:
    raise RateLimitError("Etherscan", retry_after=5)
except BlockchainError as e:
    print(f"  Caught BlockchainError: {e}")
    print(f"  Type: {type(e).__name__}")

## 5. Logging — Replacing print() in Production Code

`print()` is fine for learning, but it has serious limitations in production:
- You can't control which messages to show (turn off debug messages in prod)
- You can't write to a file and the console simultaneously
- You can't include timestamps, severity levels, or module names
- You can't distinguish "normal output" from "diagnostic information"

Python's `logging` module solves all of this.

### Log levels (from least to most severe):
```
DEBUG    — detailed info, only useful during development
INFO     — confirmation that things are working as expected
WARNING  — something unexpected but not fatal
ERROR    — a serious problem — a function failed
CRITICAL — the program cannot continue
```

### The key concept: when you set a level, you only see that level and above.
Setting `INFO` means you see INFO, WARNING, ERROR, CRITICAL — but not DEBUG.

In [ ]:
import logging

# Basic configuration — should be called once at the start of your script
logging.basicConfig(
    level   = logging.DEBUG,                          # show all levels
    format  = "%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt = "%H:%M:%S",
)

# Get a logger for this module
# Best practice: use __name__ so you know which module the message came from
logger = logging.getLogger(__name__)

# The five levels
logger.debug("Fetching block 19,847,293 from RPC")          # dev detail
logger.info("Pipeline started — processing 1,000 blocks")    # normal operation
logger.warning("RPC response slower than 500ms — may timeout")  # heads up
logger.error("Failed to fetch block 19,847,294 — skipping")     # something broke
logger.critical("Database connection lost — pipeline halted")    # fatal

print("\n(Logging output appears above with timestamps)")

In [ ]:
# Production-style logging setup — write to both console AND file

def setup_logger(name: str, log_file: str = None, level: int = logging.INFO) -> logging.Logger:
    """
    Create a logger that writes to the console and optionally to a file.

    Args:
        name:     Logger name (use __name__ in your modules)
        log_file: Optional path to a log file
        level:    Minimum log level (default: INFO)

    Returns:
        logging.Logger: Configured logger
    """
    logger    = logging.getLogger(name)
    logger.setLevel(level)

    formatter = logging.Formatter(
        "%(asctime)s | %(name)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    # Console handler — always add this
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    # File handler — optional
    if log_file:
        file_handler = logging.FileHandler(log_file)
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

    return logger


# Set up the pipeline logger
pipeline_log = setup_logger("blockchain.pipeline", log_file="pipeline.log")

# Simulate a data pipeline run
def process_block_range(start: int, end: int):
    """Simulate processing a range of blocks with proper logging."""
    pipeline_log.info(f"Starting block scan: {start:,} → {end:,} ({end-start+1} blocks)")

    processed = 0
    failed    = 0

    for block_num in range(start, end + 1):
        try:
            # Simulate: every 5th block "fails"
            if block_num % 5 == 0:
                raise ConnectionError(f"RPC timeout on block {block_num}")

            pipeline_log.debug(f"Processed block {block_num:,}")
            processed += 1

        except ConnectionError as e:
            pipeline_log.warning(f"Block {block_num:,} failed: {e} — skipping")
            failed += 1

    pipeline_log.info(
        f"Scan complete: {processed} blocks processed, {failed} failed "
        f"({failed/(processed+failed)*100:.1f}% failure rate)"
    )
    return processed, failed


p, f = process_block_range(19_847_000, 19_847_020)

## 6. Free Blockchain Data Sources — No Paid API Required

The Dune Analytics free API has been sunsetted. But there are excellent
free alternatives. This section covers what's available, what each gives you,
and how to access them — before you write a single API call (that's Week 10).

### What's available for free

| Source | What you get | Rate limit | Key needed? |
|--------|-------------|------------|-------------|
| **DefiLlama** | TVL, protocol data, yields, stablecoins | Very generous | ❌ No |
| **CoinGecko** | Token prices, market data, OHLCV history | 30 calls/min | ❌ No (basic) |
| **Etherscan** | Txns, tokens, contracts, events, ABI | 5 calls/sec | ✅ Free signup |
| **The Graph** | Indexed DeFi protocol data (Uniswap, Aave, etc.) | Generous free tier | ❌ No |
| **Ankr public RPC** | Direct Ethereum/Polygon/BSC node access | 30 req/sec | ❌ No |
| **LlamaNodes** | Multi-chain RPC endpoints | Free tier | ❌ No |
| **Blockchain.com** | Bitcoin transactions, blocks | Reasonable | ❌ No |
| **Mempool.space** | Bitcoin mempool, fee estimates | Reasonable | ❌ No |
| **OpenSea Stream** | NFT events in real-time | Limited | ✅ Free signup |

### The free data stack for this course

```
Price data    → CoinGecko (no key) or DefiLlama
Protocol TVL  → DefiLlama (no key, very comprehensive)
On-chain txns → Etherscan (free key, 30 seconds to register)
Contract data → The Graph (free, GraphQL)
RPC access    → Ankr / LlamaNodes (free, no signup)
```

This stack gives you everything you need through all 7 phases of this course.


In [ ]:
# Preview: the URLs you'll hit in Week 10 (using requests module)
# Here we just define and explain them — no HTTP calls yet

FREE_API_ENDPOINTS = {
    # DefiLlama — completely free, no key, comprehensive DeFi data
    "defillama_protocols":  "https://api.llama.fi/protocols",
    "defillama_protocol":   "https://api.llama.fi/protocol/{protocol_slug}",
    "defillama_tvl":        "https://api.llama.fi/tvl/{protocol_slug}",
    "defillama_chains":     "https://api.llama.fi/v2/chains",
    "defillama_yields":     "https://yields.llama.fi/pools",
    "defillama_stables":    "https://stablecoins.llama.fi/stablecoins",

    # CoinGecko — free tier, no key for basic endpoints
    "coingecko_markets":    "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&order=market_cap_desc",
    "coingecko_price":      "https://api.coingecko.com/api/v3/simple/price?ids={coin_id}&vs_currencies=usd",
    "coingecko_history":    "https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart?vs_currency=usd&days={days}",

    # Etherscan — free API key (register at etherscan.io/apis)
    "etherscan_txlist":     "https://api.etherscan.io/api?module=account&action=txlist&address={address}&apikey={key}",
    "etherscan_balance":    "https://api.etherscan.io/api?module=account&action=balance&address={address}&apikey={key}",
    "etherscan_tokentx":    "https://api.etherscan.io/api?module=account&action=tokentx&address={address}&apikey={key}",

    # The Graph — free, GraphQL, indexes Uniswap, Aave, Curve, etc.
    "thegraph_uniswap_v3":  "https://api.thegraph.com/subgraphs/name/uniswap/uniswap-v3",
    "thegraph_aave_v3":     "https://api.thegraph.com/subgraphs/name/aave/protocol-v3",

    # Free public RPC endpoints — no signup
    "ankr_eth_rpc":         "https://rpc.ankr.com/eth",
    "llamanodes_eth_rpc":   "https://eth.llamarpc.com",
    "publicnode_eth_rpc":   "https://ethereum.publicnode.com",
    "ankr_polygon_rpc":     "https://rpc.ankr.com/polygon",
    "ankr_arbitrum_rpc":    "https://rpc.ankr.com/arbitrum",
    "ankr_base_rpc":        "https://rpc.ankr.com/base",
}

print("Free Blockchain API Endpoints:")
print("=" * 60)
for name, url in FREE_API_ENDPOINTS.items():
    source = name.split("_")[0].upper()
    print(f"  [{source}] {name}")
    print(f"           {url[:70]}{'...' if len(url) > 70 else ''}")
    print()

In [ ]:
# Saving API endpoint config to a JSON file — a real-world pattern
# In production, you'd load this from a config file, not hardcode it

config = {
    "version": "1.0",
    "description": "Free blockchain data API configuration for the course",
    "apis": {
        "defillama": {
            "base_url":   "https://api.llama.fi",
            "requires_key": False,
            "rate_limit": "generous",
            "docs": "https://defillama.com/docs/api",
        },
        "coingecko": {
            "base_url":   "https://api.coingecko.com/api/v3",
            "requires_key": False,
            "rate_limit": "30/min on free tier",
            "docs": "https://www.coingecko.com/api/documentation",
        },
        "etherscan": {
            "base_url":   "https://api.etherscan.io/api",
            "requires_key": True,
            "key_env_var": "ETHERSCAN_API_KEY",
            "rate_limit": "5/sec on free tier",
            "signup": "https://etherscan.io/apis",
        },
        "thegraph": {
            "base_url":    "https://api.thegraph.com/subgraphs/name",
            "requires_key": False,
            "rate_limit":  "varies by subgraph",
            "docs": "https://thegraph.com/docs",
        },
    },
    "rpc_endpoints": {
        "ethereum": [
            "https://rpc.ankr.com/eth",
            "https://eth.llamarpc.com",
            "https://ethereum.publicnode.com",
        ],
        "polygon": [
            "https://rpc.ankr.com/polygon",
            "https://polygon.llamarpc.com",
        ],
        "arbitrum": [
            "https://rpc.ankr.com/arbitrum",
            "https://arbitrum.llamarpc.com",
        ],
        "base": [
            "https://rpc.ankr.com/base",
            "https://base.llamarpc.com",
        ],
    }
}

with open("api_config.json", "w") as f:
    json.dump(config, f, indent=2)

# Loading it back — how you'd use it in your scripts
with open("api_config.json") as f:
    loaded_config = json.load(f)

print("API Configuration loaded:")
for api_name, api_info in loaded_config["apis"].items():
    key_note = "✅ No key needed" if not api_info["requires_key"] else f"🔑 Key required (env: {api_info.get('key_env_var', 'N/A')})"
    print(f"  {api_name.upper():12} {key_note} | Rate limit: {api_info['rate_limit']}")

print(f"\nRPC endpoints configured for {len(loaded_config['rpc_endpoints'])} chains")

## 7. Putting It All Together — A Robust Data Pipeline

This section combines everything: CSV reading, JSON caching, error handling,
custom exceptions, and logging into a single coherent pipeline.

In [ ]:
import csv, json, os, logging, time
from collections import defaultdict

# ── Setup ─────────────────────────────────────────────────────
logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt= "%H:%M:%S",
)
log = logging.getLogger("pipeline")

class PipelineError(Exception):
    pass

class DataValidationError(PipelineError):
    def __init__(self, row_num, field, value, reason):
        self.row_num = row_num
        super().__init__(f"Row {row_num} | field='{field}' value={value!r}: {reason}")

# ── Step 1: Create sample data files ─────────────────────────
sample_rows = [
    "tx_hash,wallet,token,amount_raw,decimals,price_usd,block",
    "0xaaa,0xAlice,ETH,2500000000000000000,18,3247.85,19847293",
    "0xbbb,0xBob,USDC,5000000000,6,1.00,19847294",
    "0xccc,0xAlice,UNI,1000000000000000000000,18,12.84,19847295",
    "0xddd,0xCarol,WBTC,10000000,8,67412.0,19847296",
    "0xeee,0xBob,ETH,bad_value,18,3247.85,19847297",      # bad amount
    "0xfff,0xDave,ETH,500000000000000000,18,3247.85",     # missing block
]

with open("holdings.csv", "w") as f:
    f.write("\n".join(sample_rows))

log.info("Created sample holdings.csv")

# ── Step 2: Load and validate the CSV ─────────────────────────
def load_holdings_csv(filepath: str) -> list:
    """Load holdings CSV with validation and error recovery."""
    holdings  = []
    errors    = []

    try:
        f = open(filepath, newline="")
    except FileNotFoundError:
        raise PipelineError(f"Holdings file not found: {filepath}")

    with f:
        reader = csv.DictReader(f)
        required_fields = {"tx_hash", "wallet", "token", "amount_raw",
                           "decimals", "price_usd", "block"}

        for row_num, row in enumerate(reader, start=2):
            try:
                # Check all required fields exist
                missing = required_fields - set(row.keys())
                if missing:
                    raise DataValidationError(row_num, "N/A", "N/A",
                                              f"missing fields: {missing}")

                holdings.append({
                    "tx_hash":    row["tx_hash"].strip(),
                    "wallet":     row["wallet"].strip(),
                    "token":      row["token"].strip(),
                    "amount_raw": int(row["amount_raw"]),           # raises ValueError if bad
                    "decimals":   int(row["decimals"]),
                    "price_usd":  float(row["price_usd"]),
                    "block":      int(row["block"]),
                })

            except (ValueError, KeyError) as e:
                err = DataValidationError(row_num, "unknown", "unknown", str(e))
                log.warning(str(err))
                errors.append(err)
                continue
            except DataValidationError as e:
                log.warning(str(e))
                errors.append(e)
                continue

    log.info(f"Loaded {len(holdings)} valid rows, {len(errors)} rows had errors")
    return holdings


holdings = load_holdings_csv("holdings.csv")

# ── Step 3: Enrich with human-readable amounts ─────────────────
for h in holdings:
    h["amount_human"] = h["amount_raw"] / 10**h["decimals"]
    h["value_usd"]    = h["amount_human"] * h["price_usd"]

# ── Step 4: Aggregate by wallet ────────────────────────────────
wallet_totals = defaultdict(float)
for h in holdings:
    wallet_totals[h["wallet"]] += h["value_usd"]

# ── Step 5: Write enriched output ─────────────────────────────
output_fields = ["tx_hash","wallet","token","amount_human","value_usd","block"]
with open("holdings_enriched.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=output_fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(holdings)

log.info("Written holdings_enriched.csv")

# ── Step 6: Cache summary as JSON ─────────────────────────────
summary = {
    "generated_at":  int(time.time()),
    "total_wallets": len(wallet_totals),
    "total_records": len(holdings),
    "wallet_values": {w: round(v, 2) for w, v in
                      sorted(wallet_totals.items(), key=lambda x: -x[1])},
}

with open("summary.json", "w") as f:
    json.dump(summary, f, indent=2)

log.info("Written summary.json")

print("\n── Pipeline complete ──")
print(f"Wallet portfolio values:")
for wallet, value in summary["wallet_values"].items():
    print(f"  {wallet:<10} ${value:>10,.2f}")

## Summary

| Concept | Key function / pattern | Blockchain use |
|---------|----------------------|----------------|
| Write CSV | `csv.DictWriter` | Save transaction exports, analysis results |
| Read CSV | `csv.DictReader` + type casting | Load Etherscan/exchange exports |
| Write JSON | `json.dump(data, f, indent=2)` | Cache API responses, save config |
| Read JSON | `json.load(f)` | Parse API responses |
| Cache pattern | Check file age with `os.path.getmtime()` | Avoid re-fetching during dev |
| `try/except` | Specific exception types | Handle RPC timeouts, bad data |
| `else` | Runs on success | Process result only if no error |
| `finally` | Always runs | Close files, log timing |
| Custom exceptions | `class MyError(Exception)` | `InvalidAddressError`, `APIError` |
| Logging | `logging.getLogger(__name__)` | Pipeline monitoring, debug vs prod |
| Free APIs | DefiLlama, CoinGecko, Etherscan | No paid key needed for this course |

---

## What's next

**Week 8 — Pandas & NumPy:** Load your CSVs into DataFrames, filter rows
like WHERE clauses, aggregate like GROUP BY, and join like SQL JOINs.
The SQL skills you already have transfer directly to Pandas syntax.

---

**Your task before Week 8:**
1. Complete `exercises.py` — build your own robust CSV/JSON pipeline
2. Register for a **free Etherscan API key** at `etherscan.io/apis`
   (takes 30 seconds — you'll need it for Week 10)
3. Save your key to a `.env` file (never commit this to GitHub):
   ```bash
   echo "ETHERSCAN_API_KEY=your_key_here" >> .env
   echo ".env" >> .gitignore
   ```
4. Commit:
```bash
git add phase-2-python-for-data/week-07-files-error-handling/
git commit -m "phase-2/week-07: file handling and error handling — lesson + exercises"
git push
```